# Progetto Machine Learning: Previsione del Prezzo Unico Nazionale (PUN)
## 4. Modeling & 5. Evaluation (Multi-Horizon Forecasting)

**Corso:** Machine Learning e Data Mining (Modulo 2) — Prof. Matteo Francia  
**Dataset:** Prezzi orari MGP (GME) & Dati Meteo Zonali (Open-Meteo) 2024–2025  

---

### Obiettivi e Metodologia CRISP-DM

Questo notebook copre la seconda parte del progetto, focalizzata sulle fasi **4. Modeling** e **5. Evaluation** della metodologia CRISP-DM:

1. **Integrazione dei Dataset Processati:** Importazione delle matrici pulite (`X_train`, `y_train`, `X_test`, `y_test`) generate nella fase di Data Preparation (`01_data_ingestion.ipynb`).
2. **Definizione delle Baseline:** Addestramento di modelli di riferimento (Naive e Regressione Ridge/Lasso) per definire la soglia minima di performance accettabile.
3. **Modellazione Predittiva Avanzata:** Implementazione e benchmarking di algoritmi di Machine Learning (Decision Tree, Random Forest, XGBoost, MLP) per previsioni multi-orizzonte (24h, 48h, 72h).
4. **Valutazione delle Performance:** Analisi e confronto delle metriche di errore ($\text{MAE}$, $\text{RMSE}$, $\text{MAPE}$) sul test set per identificare il modello con il miglior grado di generalizzazione.

In [27]:
# --- SETUP AMBIENTE E IMPORT LIBRERIE ---
import os
import warnings
warnings.filterwarnings('ignore')

# Gestione Working Directory (Google Colab vs Locale Mac)
if os.path.exists("/content"):
    if not os.path.exists("progetto-ml-pun"):
        !git clone https://github.com/erloyz/progetto-ml-pun.git
    %cd /content/progetto-ml-pun
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        %cd ..

print("📍 Working Directory corrente:", os.getcwd())

# --- MANIPOLAZIONE DATI E GRAFICI ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- MACHINE LEARNING (Scikit-Learn) ---
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

# Metriche di valutazione della regressione
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

print("Ambiente configurato e librerie caricate con successo!")

📍 Working Directory corrente: /Users/erloyz/Desktop/uni/dm-ml/ml/progetto-ml-pun
Ambiente configurato e librerie caricate con successo!


## 1. Caricamento Dataset Processati
Caricamento dei dataset di Training e Test (`X_train`, `X_test`, `y_train`, `y_test`) esportati dalla pipeline di ingestion[cite: 9, 10]. Il data split è stato effettuato in ordine cronologico per evitare Data Leakage[cite: 9].

In [28]:
# --- CARICAMENTO DATASET PROCESSATI ---

X_train = pd.read_csv("data/processed/X_train.csv")
y_train = pd.read_csv("data/processed/y_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")
y_test = pd.read_csv("data/processed/y_test.csv")

# Conversione di y in array 1D per evitare warning di Scikit-Learn durante il .fit()
y_train_vec = y_train.values.ravel()
y_test_vec = y_test.values.ravel()

print(f"Dati caricati correttamente!")
print(f" - X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f" - X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print(f" - X_train nulli: {X_train.isnull().sum().sum()} | y_train nulli: {np.isnan(y_train_vec).sum()}")
print(f" - X_test nulli:  {X_test.isnull().sum().sum()} | y_test nulli:  {np.isnan(y_test_vec).sum()}")

Dati caricati correttamente!
 - X_train shape: (12959, 21) | y_train shape: (12959, 1)
 - X_test shape:  (4417, 21)  | y_test shape:  (4417, 1)
 - X_train nulli: 0 | y_train nulli: 0
 - X_test nulli:  0 | y_test nulli:  0


In [29]:
# --- VERIFICA STRUTTURA DELLE FEATURE ---
print(f"Numero totale di Feature di input (N): {X_train.shape[1]}\n")
print("Lista delle variabili di input:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")

Numero totale di Feature di input (N): 21

Lista delle variabili di input:
   1. Data
   2. Ora
   3. temperature_2m_CentroSud
   4. temperature_2m_Nord
   5. wind_speed_10m_CentroSud
   6. wind_speed_10m_Nord
   7. shortwave_radiation_CentroSud
   8. shortwave_radiation_Nord
   9. ora_num
  10. mese_num
  11. giorno_settimana
  12. sin_ora
  13. cos_ora
  14. sin_mese
  15. cos_mese
  16. is_weekend
  17. pun_lag_24
  18. pun_lag_48
  19. pun_lag_72
  20. pun_lag_168
  21. pun_rolling_mean_24h


## 2. Feature Scaling (Prevenzione Data Leakage)
Applicazione del **MinMaxScaler [0, 1]** per le feature numeriche destinate a modelli sensibili alla scala (Regressione Lineare, Ridge, Lasso, MLPRegressor)[cite: 9]. 
> **Nota Anti-Leakage:** Il metodo `.fit()` viene calcolato **esclusivamente su `X_train`**; su `X_test` viene applicato soltanto il `.transform()` usando i minimi e massimi appresi dallo storico[cite: 9].

In [30]:
# ==============================================================================
# FEATURE SCALING (MinMaxScaler per modelli sensibili alla scala)
# ==============================================================================
# Applico MinMaxScaler [0, 1] per normalizzare le feature per modelli come Ridge e MLPRegressor.
# Il .fit() viene eseguito ESCLUSIVAMENTE su X_train per imparare min e max storici,
# mentre su X_test si applica soltanto il .transform() per prevenire il Data Leakage.

from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. Escludiamo le colonne di tipo data/stringa prima dello scaling
cols_to_drop = [col for col in ['Data', 'timestamp'] if col in X_train.columns]

X_train_num = X_train.drop(columns=cols_to_drop)
X_test_num = X_test.drop(columns=cols_to_drop)

# 2. Inizializzazione dello Scaler per le feature numeriche
scaler_X = MinMaxScaler(feature_range=(0, 1))

# 3. Fit su X_train numerico | Transform sia su Train che su Test
X_train_scaled = scaler_X.fit_transform(X_train_num)
X_test_scaled = scaler_X.transform(X_test_num)

# 4. Conversione in DataFrame mantenendo i nomi delle colonne
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_num.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_num.columns)

print("✅ Feature Scaling (MinMaxScaler) completato correttamente!")
print(f"Dimensioni X_train scalato: {X_train_scaled.shape}")
print(f"Dimensioni X_test scalato:  {X_test_scaled.shape}")

✅ Feature Scaling (MinMaxScaler) completato correttamente!
Dimensioni X_train scalato: (12959, 20)
Dimensioni X_test scalato:  (4417, 20)


## 3. Definizione del Framework di Valutazione
Definizione della funzione `calcola_metriche()` per la registrazione unificata delle prestazioni del modello sui prezzi reali in €/MWh tramite **MAE**, **RMSE** e **MAPE**[cite: 9].

In [31]:
# --- DEFINIZIONE FUNZIONE DI VALUTAZIONE METRICHE ---
def calcola_metriche(y_true, y_pred, nome_modello="Modello"):
    """
    Calcola e restituisce MAE, RMSE e MAPE per un dato modello.
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    print(f"Performance {nome_modello}:")
    print(f"MAE:  {mae:.4f} €/MWh")
    print(f"RMSE: {rmse:.4f} €/MWh")
    print(f"MAPE: {mape:.2f} %")
    
    return {"Modello": nome_modello, "MAE": mae, "RMSE": rmse, "MAPE": mape}

# Lista per collezionare i risultati dei vari modelli e confrontarli alla fine
risultati_benchmark = []

## 4. Modeling & Benchmarking

### 4.0 Baseline Model (Persistence Lag-24)
Modello di riferimento naïve: ipotizza che il prezzo all'ora $t$ sia uguale a quello del giorno precedente alla stessa ora ($t-24$)[cite: 9]. Serve come soglia minima di accuratezza per validare i modelli ML[cite: 9].

In [32]:
# ==============================================================================
# MODELLO 0: BASELINE DI PERSISTENZA (LAG-24)
# ==============================================================================
# La baseline ipotizza che il prezzo all'ora t sia uguale a quello del giorno prima alla stessa ora (t-24).

# Utilizziamo la colonna 'pun_lag_24' presente in X_test
y_pred_baseline = X_test['pun_lag_24']

# Calcolo metriche per la Baseline
res_baseline = calcola_metriche(y_test_vec, y_pred_baseline, nome_modello="Baseline (Persistence Lag-24)")
risultati_benchmark.append(res_baseline)

Performance Baseline (Persistence Lag-24):
MAE:  10.3154 €/MWh
RMSE: 16.1012 €/MWh
MAPE: 11.79 %


### 4.1 Modelli Statistici e Lineari (Ridge & Lasso Regression)
Addestramento dei modelli di regressione lineare regolarizzata con penalità $L_2$ (Ridge) ed $L_1$ (Lasso) utilizzando le matrici di feature scalate (`X_train_scaled`, `X_test_scaled`) per prevenire squilibri tra le variabili esogene e le feature cicliche.

* **Ridge ($L_2$):** Applica una penalità quadratica sui pesi, riducendoli in modo uniforme senza annullarli del tutto.
* **Lasso ($L_1$):** Applica una penalità sul valore assoluto dei coefficienti, azzerando le variabili meno rilevanti e fungendo da metodo integrato di **Feature Selection automatica**.

In [41]:
# ==============================================================================
# MODELLO 1: RIDGE REGRESSION (REGOLARIZZAZIONE L2)
# ==============================================================================
from sklearn.linear_model import Ridge

# 1. Inizializzazione e fit del modello sui dati SCALATI
model_ridge = Ridge(alpha=1.0, random_state=42)
model_ridge.fit(X_train_scaled, y_train_vec)

# 2. Previsione sul Test Set scalato
y_pred_ridge = model_ridge.predict(X_test_scaled)

# 3. Calcolo e salvataggio delle metriche
res_ridge = calcola_metriche(y_test_vec, y_pred_ridge, nome_modello="Ridge Regression")
risultati_benchmark.append(res_ridge)

Performance Ridge Regression:
MAE:  8.6835 €/MWh
RMSE: 11.9153 €/MWh
MAPE: 9.81 %


In [42]:
# ==============================================================================
# MODELLO 1B: LASSO REGRESSION (REGOLARIZZAZIONE L1)
# ==============================================================================
from sklearn.linear_model import Lasso

model_lasso = Lasso(alpha=0.01, random_state=42)
model_lasso.fit(X_train_scaled, y_train_vec)

y_pred_lasso = model_lasso.predict(X_test_scaled)

res_lasso = calcola_metriche(y_test_vec, y_pred_lasso, nome_modello="Lasso Regression")
risultati_benchmark.append(res_lasso)

Performance Lasso Regression:
MAE:  8.6472 €/MWh
RMSE: 11.8798 €/MWh
MAPE: 9.78 %


### 4.2 Modelli Tree-Based ed Ensemble Learning (Random Forest & XGBoost)
Addestramento di modelli non parametrici basati su alberi decisionali (Random Forest e XGBoost)[cite: 9, 13]. Gli alberi gestiscono nativamente relazioni non lineari, interazioni complesse tra le feature e sono invarianti rispetto allo scaling numerico delle variabili[cite: 9, 13].

In [37]:
# ==============================================================================
# MODELLO 2: RANDOM FOREST REGRESSOR
# ==============================================================================
from sklearn.ensemble import RandomForestRegressor

# 1. Utilizziamo X_train_num e X_test_num (senza la colonna della data)
model_rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
model_rf.fit(X_train_num, y_train_vec)

# 2. Previsione sul Test Set
y_pred_rf = model_rf.predict(X_test_num)

# 3. Calcolo e salvataggio delle metriche
res_rf = calcola_metriche(y_test_vec, y_pred_rf, nome_modello="Random Forest")
risultati_benchmark.append(res_rf)

Performance Random Forest:
MAE:  8.1320 €/MWh
RMSE: 11.9628 €/MWh
MAPE: 9.39 %


In [50]:
# ==============================================================================
# XGBOOST HYPERPARAMETER TUNING (TIMESERIES CROSS-VALIDATION)
# ==============================================================================
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# 1. Definizione della griglia degli iperparametri
param_dist = {
    'n_estimators': [150, 300, 500],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0.0, 0.1, 1.0],
    'reg_lambda': [1.0, 2.0, 5.0]
}

# 2. TimeSeriesSplit per rispettare la sequenza temporale ed evitare Data Leakage
tscv = TimeSeriesSplit(n_splits=3)

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,  # 15 combinazioni casuali per limitare i tempi di calcolo
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1
)

# 3. Fit sulle matrici numeri di addestramento (X_train_num, y_train_vec)
xgb_search.fit(X_train_num, y_train_vec)

print("🏆 Migliori Iperparametri trovati per XGBoost:")
print(xgb_search.best_params_)

# 4. Valutazione del miglior modello sul Test set
model_xgb_tuned = xgb_search.best_estimator_
y_pred_xgb_tuned = model_xgb_tuned.predict(X_test_num)

res_xgb_tuned = calcola_metriche(y_test_vec, y_pred_xgb_tuned, nome_modello="XGBoost (Tuned)")
print(res_xgb_tuned)

🏆 Migliori Iperparametri trovati per XGBoost:
{'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
Performance XGBoost (Tuned):
MAE:  7.4375 €/MWh
RMSE: 11.0131 €/MWh
MAPE: 8.53 %
{'Modello': 'XGBoost (Tuned)', 'MAE': 7.437476720050573, 'RMSE': np.float64(11.013114638741717), 'MAPE': 8.534423902006655}


### 4.3 Deep Learning / Reti Neurali (MLPRegressor)
Addestramento di una Rete Neurale Multi-Layer Perceptron (MLP) sulle matrici di feature scalate (`X_train_scaled`, `X_test_scaled`). Le reti neurali richiedono dati normalizzati in scala [0, 1] per garantire la stabilità numerica e la rapida convergenza della discesa del gradiente (algoritmo Adam)[cite: 10].

In [39]:
# ==============================================================================
# MODELLO 4: MULTI-LAYER PERCEPTRON (MLP REGRESSOR)
# ==============================================================================
from sklearn.neural_network import MLPRegressor

# 1. Inizializzazione della Rete Neurale con 2 hidden layer (64 e 32 neuroni)
model_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,  # Previene l'overfitting interrompendo l'addestramento se l'errore non migliora
    n_iter_no_change=10
)

# 2. Fit sui dati SCALATI
model_mlp.fit(X_train_scaled, y_train_vec)

# 3. Previsione sul Test Set scalato
y_pred_mlp = model_mlp.predict(X_test_scaled)

# 4. Calcolo e salvataggio delle metriche
res_mlp = calcola_metriche(y_test_vec, y_pred_mlp, nome_modello="MLP Regressor")
risultati_benchmark.append(res_mlp)

Performance MLP Regressor:
MAE:  8.4655 €/MWh
RMSE: 11.8834 €/MWh
MAPE: 9.75 %


### 4.4 Support Vector Regression (SVR)
Addestramento del modello **Support Vector Regressor** con kernel RBF (Radial Basis Function) sulle matrici di feature scalate (`X_train_scaled`, `X_test_scaled`).

La funzione di perdita $\epsilon$-insensitive dell'SVR rende l'algoritmo particolarmente robusto ai picchi isolati di prezzo e al rumore di fondo dei mercati energetici. Come evidenziato dalla letteratura di settore, le Support Vector Machines si posizionano costantemente tra le architetture top-performer per la previsione del PUN nel Day-Ahead Market.

In [43]:
# ==============================================================================
# MODELLO 5: SUPPORT VECTOR REGRESSION (SVR RBF)
# ==============================================================================
from sklearn.svm import SVR

# 1. Inizializzazione SVR con kernel RBF (C=10 per bilanciare margine e errore)
model_svr = SVR(kernel='rbf', C=10.0, epsilon=0.1)

# 2. Fit sui dati SCALATI
model_svr.fit(X_train_scaled, y_train_vec)

# 3. Previsione sul Test Set scalato
y_pred_svr = model_svr.predict(X_test_scaled)

# 4. Calcolo e salvataggio delle metriche
res_svr = calcola_metriche(y_test_vec, y_pred_svr, nome_modello="SVR (RBF Kernel)")
risultati_benchmark.append(res_svr)

Performance SVR (RBF Kernel):
MAE:  8.5119 €/MWh
RMSE: 12.0670 €/MWh
MAPE: 10.15 %


In [49]:
# ==============================================================================
# MODELLO 5: LINEAR SUPPORT VECTOR REGRESSION (OPTIMIZED LINEARSVR)
# ==============================================================================
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler

# 1. Scaling con StandardScaler (fondamentale per la geometria lineare dell'SVR)
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train_num)
X_test_std = scaler_std.transform(X_test_num)

# 2. Inizializzazione LinearSVR:
# - C alto (100.0 o 200.0) per catturare i picchi di prezzo ed evitare underfitting
# - epsilon piccolo (0.01) per ridurre il tubo di tolleranza dell'errore
# - max_iter aumentato a 10.000 per garantire la convergenza della discesa del gradiente
model_svr_linear = LinearSVR(
    C=500.0, 
    epsilon=0.01, 
    random_state=42, 
    max_iter=10000
)

# 3. Fit sui dati STANDARDIZZATI
model_svr_linear.fit(X_train_std, y_train_vec)

# 4. Previsione e metriche sul Test Set
y_pred_svr_linear = model_svr_linear.predict(X_test_std)

res_svr_linear = calcola_metriche(y_test_vec, y_pred_svr_linear, nome_modello="SVR (Linear Kernel)")

# Aggiornamento/Salvataggio nei risultati
risultati_benchmark.append(res_svr_linear)

Performance SVR (Linear Kernel):
MAE:  8.2426 €/MWh
RMSE: 11.8769 €/MWh
MAPE: 9.59 %
